# Stage 6A — Acquisition Universe and Source Audit

This notebook freezes the strategy-and-company-result acquisition universe, records discovered source families, applies conservative access/copyright/redistribution treatment, and validates readiness for controlled official-source acquisition.

It does not calculate strategy effectiveness, create strategy-result links, rank companies, create a composite score, select an overall winner, or prepare a report/README.

## Environment Setup

Import libraries and define the locked Stage 5 commit, expected input checksums, deterministic output root, and retrieval date.

In [1]:
from __future__ import annotations
import hashlib, os, shutil, urllib.error, urllib.parse, urllib.request
from pathlib import Path
import pandas as pd

REPOSITORY_FULL_NAME = "Ronaldo-spec/indonesia-fmcg-brand-portfolio-analysis"
INPUT_COMMIT = "22a5779bcd5594eb74dee162a18010121156350e"
RETRIEVAL_DATE = "2026-08-21"
INPUT_LOCKS = {'metadata/stage5_input_lock.csv': 'd818327e572dc52c4f74db9231f9207383a39a926e425ba13d25919341ce58da', 'metadata/stage5_research_questions.csv': '6f2854d93713534e89f64ff743d4ec2a0bdd101e9a109989cb13bfcecd1aaa14', 'metadata/stage5_period_scope.csv': '7c70d6e7c6b1f917ad3685c08f5f0b2d215e7ac62b008b5e7cee0bfec0f3b713', 'metadata/stage5_strategy_taxonomy.csv': '65c5ffd83c847b9f93c64a4bafd8ca6bf27abe26ee5ea2acd2ff69e07862c684', 'metadata/stage5_outcome_taxonomy.csv': '32a3b729ea1e903a24a30e5927e47e909745e527c42281dd0997a7919320cc03', 'metadata/stage5_reporting_entity_scope.csv': '6db1c9e177363b9e789be2094d1f922a1a9fcfb7b4d659fe1c3ee71a1be508e8', 'metadata/stage5_source_requirements.csv': 'bc644cc6dc5fa6ad5fa8a7df7799e1ff5a4512e4beeb0baff08fd5ac3c053c2c', 'metadata/stage5_attribution_evidence_rules.csv': 'a2afb7e170f71d6bb6937cfc41c3e84c027c4e2df82525096a2cf776215006a5', 'metadata/stage5_linkage_eligibility_rules.csv': '39142fd09351f473c033a8862678e5ff1327871407e05a8b1e6b9093c74b092a', 'metadata/stage5_design_validation.csv': 'bdff4b05d8f23a1f5f53b8cfd31cfb8f218d3e831a7c70f375403bdbc9be79e0'}
OUTPUT_ROOT = Path(os.environ.get("FMCG_STAGE6A_OUTPUT_ROOT", "/content/fmcg_stage6a_outputs"))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
print(f"Locked input commit: {INPUT_COMMIT}")
print(f"Required Stage 5 inputs: {len(INPUT_LOCKS)}")
print(f"Output root: {OUTPUT_ROOT}")

Locked input commit: 22a5779bcd5594eb74dee162a18010121156350e
Required Stage 5 inputs: 10
Output root: /content/fmcg_stage6a_outputs


## Locked Stage 5 Input Retrieval

Retrieve only the ten manifest-tracked Stage 5 metadata artifacts from the authoritative commit. Colab uses the `GITHUB_TOKEN` secret; a local validation root can be supplied through an environment variable.

In [2]:
configured_root = os.environ.get("FMCG_STAGE6A_INPUT_ROOT")
if configured_root:
    INPUT_ROOT = Path(configured_root)
    input_mode = "local_validation_root"
else:
    try:
        from google.colab import userdata
    except ImportError as exc:
        raise RuntimeError("Run in Google Colab or set FMCG_STAGE6A_INPUT_ROOT for local validation.") from exc
    github_token = userdata.get("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError("Colab Secret GITHUB_TOKEN is unavailable or access has not been granted.")
    INPUT_ROOT = Path("/content/fmcg_stage6a_inputs")
    if INPUT_ROOT.exists():
        shutil.rmtree(INPUT_ROOT)
    INPUT_ROOT.mkdir(parents=True, exist_ok=True)
    for relative_path in INPUT_LOCKS:
        encoded_path = urllib.parse.quote(relative_path, safe="/")
        url = f"https://api.github.com/repos/{REPOSITORY_FULL_NAME}/contents/{encoded_path}?ref={INPUT_COMMIT}"
        request = urllib.request.Request(url, headers={"Authorization": f"Bearer {github_token}", "Accept": "application/vnd.github.raw+json", "X-GitHub-Api-Version": "2022-11-28", "User-Agent": "fmcg-stage6a-colab"})
        destination = INPUT_ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                destination.write_bytes(response.read())
        except urllib.error.HTTPError as exc:
            raise RuntimeError(f"GitHub input retrieval failed for {relative_path} with HTTP {exc.code}.") from exc
    del github_token
    input_mode = "locked_github_commit"
missing = [p for p in INPUT_LOCKS if not (INPUT_ROOT / p).exists()]
if missing:
    raise FileNotFoundError(f"Missing required Stage 5 inputs: {missing}")
print(f"Input mode: {input_mode}")
print(f"Required files found: {len(INPUT_LOCKS)}/{len(INPUT_LOCKS)}")

Input mode: locked_github_commit
Required files found: 10/10


## Input Integrity and Prior-Stage Gate

Verify all inherited Stage 5 artifacts against their locked SHA-256 values and confirm the final Stage 5 gate before any acquisition metadata is generated.

In [3]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

lock_rows = []
for relative_path, expected_sha256 in INPUT_LOCKS.items():
    actual_sha256 = sha256_file(INPUT_ROOT / relative_path)
    lock_rows.append({"file_path": relative_path, "expected_sha256": expected_sha256, "actual_sha256": actual_sha256, "hash_match": actual_sha256 == expected_sha256, "locked_repository_commit": INPUT_COMMIT})
stage6a_input_lock = pd.DataFrame(lock_rows)
if not stage6a_input_lock["hash_match"].all():
    raise RuntimeError("One or more Stage 5 input checksums failed.")
stage5_validation = pd.read_csv(INPUT_ROOT / "metadata/stage5_design_validation.csv", dtype=str, keep_default_na=False)
final_stage5 = stage5_validation.loc[stage5_validation["check_id"] == "S5D026"].iloc[0]
if final_stage5["result"] != "PASS_WITH_CAVEAT" or final_stage5["status"] != "passed_with_caveat":
    raise RuntimeError("Stage 5 final gate is not the expected PASS_WITH_CAVEAT status.")
print(f"Input checksums passed: {stage6a_input_lock['hash_match'].sum()}/{len(stage6a_input_lock)}")
print(f"Stage 5 gate: {final_stage5['result']} / {final_stage5['status']}")

Input checksums passed: 10/10
Stage 5 gate: PASS_WITH_CAVEAT / passed_with_caveat


## Frozen Acquisition Universe

Define the company/entity/source universe before acquisition. Missing public disclosure remains an explicit evidence state rather than a performance value.

In [4]:
acquisition_columns = ["target_id","canonical_group","reporting_perimeter","portfolio_role","evidence_family","target_source_type","target_period","geography","strategy_codes","outcome_ids","discovery_status","required_treatment"]
stage6a_acquisition_universe = pd.DataFrame([('S6A_T01', 'Wings Group', 'Wings Group strict-control consumer portfolio', 'primary_portfolio', 'strategy_events', 'official_corporate_or_brand_announcement', '2022–2025', 'Indonesia', 'STR01;STR02;STR03;STR04;STR05;STR06;STR07;STR08;STR09', 'OUT07;OUT08;OUT09;OUT10;OUT15;OUT16;OUT17', 'discovered', 'Acquire dated official actions only; do not infer effectiveness.'), ('S6A_T02', 'Wings Group', 'Wings Group strict-control consumer portfolio', 'primary_portfolio', 'current_portfolio_context', 'official_corporate_webpage', 'current_page', 'Indonesia', 'STR01;STR04;STR09', 'OUT01', 'discovered', 'Use only for current portfolio and operating-scope context; do not backdate current pages.'), ('S6A_T03', 'Wings Group', 'Wings Group strict-control consumer portfolio', 'primary_portfolio', 'annual_reporting', 'annual_report', 'FY2022–FY2025', 'Indonesia', '', 'OUT07;OUT08;OUT11;OUT12;OUT13;OUT14;OUT15;OUT16', 'not_found_in_official_site_discovery', 'Retain as a disclosure limitation; do not substitute unofficial estimates as equivalent evidence.'), ('S6A_T04', 'Wings Group', 'Wings Group strict-control consumer portfolio', 'primary_portfolio', 'audited_financials', 'audited_financial_statement', 'FY2022–FY2025', 'Indonesia', '', 'OUT07;OUT08;OUT11;OUT12;OUT13;OUT14;OUT15', 'not_found_in_official_site_discovery', 'Retain as a disclosure limitation; missing financial disclosure is not zero or weak performance.'), ('S6A_T05', 'Wings Group', 'PT Wings Surya / PT Sayap Mas Utama where required', 'attribution_resolution', 'operating_entity_resolution', 'official_corporate_webpage_or_announcement', '2022–2025', 'Indonesia', 'STR01;STR03;STR04;STR06;STR09', 'OUT07;OUT08;OUT15;OUT16;OUT17', 'partially_discovered', 'Resolve exact operating entity only where a result attribution requires it.'), ('S6A_T06', 'Indofood', 'PT Indofood Sukses Makmur Tbk', 'parent_company', 'annual_reporting', 'annual_report', 'FY2022–FY2025', 'Indonesia / consolidated with segment disclosures', 'STR01;STR04;STR05;STR06;STR07;STR08;STR09', 'OUT07;OUT08;OUT11;OUT12;OUT13;OUT14;OUT15;OUT16;OUT17', 'discovered', 'Preserve consolidated and segment boundaries.'), ('S6A_T07', 'Indofood', 'PT Indofood Sukses Makmur Tbk', 'parent_company', 'audited_financials', 'audited_financial_statement', 'FY2022–FY2025', 'Indonesia / consolidated', '', 'OUT07;OUT08;OUT11;OUT12;OUT13;OUT14;OUT15', 'discovered', 'Do not attribute parent consolidated results to individual packaged-FMCG brands.'), ('S6A_T08', 'Indofood', 'PT Indofood Sukses Makmur Tbk', 'parent_company', 'period_results', 'official_earnings_release', 'FY2022–FY2025', 'Indonesia / consolidated with disclosed segments', 'STR04;STR05;STR06;STR08', 'OUT07;OUT08;OUT09;OUT10;OUT13;OUT14', 'discovered', 'Reconcile headline measures with annual or audited reporting.'), ('S6A_T09', 'Indofood', 'PT Indofood CBP Sukses Makmur Tbk', 'primary_portfolio', 'annual_reporting', 'annual_report', 'FY2022–FY2025', 'Indonesia plus disclosed international operations', 'STR01;STR02;STR03;STR04;STR05;STR06;STR07;STR08;STR09', 'OUT07;OUT08;OUT09;OUT10;OUT11;OUT12;OUT13;OUT14;OUT15;OUT16;OUT17', 'discovered', 'Preserve ICBP reporting perimeter and geography.'), ('S6A_T10', 'Indofood', 'PT Indofood CBP Sukses Makmur Tbk', 'primary_portfolio', 'audited_financials', 'audited_financial_statement', 'FY2022–FY2025', 'consolidated', '', 'OUT07;OUT08;OUT11;OUT12;OUT13;OUT14;OUT15', 'discovered', 'Use consolidated ICBP results only at the disclosed perimeter.'), ('S6A_T11', 'Indofood', 'PT Indofood CBP Sukses Makmur Tbk', 'primary_portfolio', 'period_results', 'official_earnings_release', 'FY2022–FY2025', 'consolidated', 'STR02;STR04;STR05;STR06;STR08', 'OUT07;OUT08;OUT09;OUT10;OUT13;OUT14', 'discovered', 'Keep management explanations labelled company-reported.'), ('S6A_T12', 'Indofood', 'Bogasari consumer flour portfolio', 'primary_portfolio', 'segment_evidence', 'annual_report_or_segment_disclosure', 'FY2022–FY2025', 'Indonesia', 'STR01;STR02;STR04;STR05;STR06;STR07;STR08', 'OUT07;OUT08;OUT09;OUT15;OUT16;OUT17', 'discovered', 'Exclude industrial-only outcomes from packaged-FMCG interpretation.'), ('S6A_T13', 'Indofood', 'Agribusiness consumer edible oils and fats', 'primary_portfolio', 'segment_evidence', 'annual_report_or_segment_disclosure', 'FY2022–FY2025', 'Indonesia', 'STR01;STR02;STR04;STR05;STR06;STR07;STR08', 'OUT07;OUT08;OUT09;OUT15;OUT16;OUT17', 'discovered', 'Separate consumer oils and fats from plantations and upstream operations.'), ('S6A_T14', 'Mayora', 'PT Mayora Indah Tbk and consolidated subsidiaries', 'primary_portfolio', 'annual_reporting', 'annual_report', 'FY2022–FY2025', 'Indonesia plus exports', 'STR01;STR02;STR03;STR04;STR05;STR06;STR07;STR08;STR09', 'OUT07;OUT08;OUT09;OUT10;OUT11;OUT12;OUT13;OUT14;OUT15;OUT16;OUT17', 'discovered', 'Preserve domestic/export and consolidated scope.'), ('S6A_T15', 'Mayora', 'PT Mayora Indah Tbk and consolidated subsidiaries', 'primary_portfolio', 'audited_financials', 'audited_financial_statement', 'FY2022–FY2025', 'Indonesia plus exports', '', 'OUT07;OUT08;OUT11;OUT12;OUT13;OUT14;OUT15', 'discovered', 'Do not interpret export-heavy consolidated results as Indonesian household demand.'), ('S6A_T16', 'Mayora', 'PT Mayora Indah Tbk', 'primary_portfolio', 'public_disclosures', 'regulatory_or_public_expose', '2022–2025', 'Indonesia / disclosed transaction scope', 'STR06;STR07;STR09', 'OUT15;OUT16', 'discovered_with_period_gaps', 'Use only dated disclosures with explicit transaction/entity scope.'), ('S6A_T17', 'Mayora', 'PT Mayora Indah Tbk and consolidated subsidiaries', 'primary_portfolio', 'geography_split', 'annual_report_or_financial_statement', 'FY2022–FY2025', 'domestic versus export', '', 'OUT07;OUT08', 'discovered', 'Retain domestic and export measures separately where disclosed.'), ('S6A_T18', 'Mayora', 'PT Tirta Fresindo Jaya / Le Minerale', 'sensitivity_only', 'affiliate_context', 'official_or_authoritative_source', '2022–2025', 'Indonesia', 'STR01;STR03;STR04;STR06;STR09', 'OUT07;OUT08;OUT16;OUT17', 'conditional_sensitivity_only', 'Do not include in strict-control primary Mayora results.'), ('S6A_T19', 'Unilever Indonesia', 'PT Unilever Indonesia Tbk controlled portfolio', 'primary_time_varying', 'annual_reporting', 'annual_report', 'FY2022–FY2025', 'Indonesia', 'STR01;STR02;STR03;STR04;STR05;STR06;STR07;STR08;STR09', 'OUT07;OUT08;OUT09;OUT10;OUT11;OUT12;OUT13;OUT14;OUT15;OUT16;OUT17;OUT18', 'discovered', 'Apply ownership-period validity to every brand/business result.'), ('S6A_T20', 'Unilever Indonesia', 'PT Unilever Indonesia Tbk controlled portfolio', 'primary_time_varying', 'audited_financials', 'audited_financial_statement', 'FY2022–FY2025', 'Indonesia with export disclosure where reported', '', 'OUT07;OUT08;OUT11;OUT12;OUT13;OUT14;OUT15', 'discovered', 'Preserve restatements, continuing-operation scope and geography.'), ('S6A_T21', 'Unilever Indonesia', 'PT Unilever Indonesia Tbk controlled portfolio', 'primary_time_varying', 'results_presentations', 'official_investor_presentation', '2022–2025', 'Indonesia', 'STR01;STR02;STR03;STR04;STR05;STR06;STR07;STR08', 'OUT07;OUT08;OUT09;OUT10;OUT13;OUT14;OUT17;OUT18', 'discovered', 'Treat strategy-result explanations as company-reported unless independently supported.'), ('S6A_T22', 'Unilever Indonesia', 'Ice Cream and SariWangi disposal scope', 'ownership_change_context', 'disposal_filings_and_financials', '2024–2026', 'Indonesia', 'STR09', 'OUT07;OUT08;OUT13;OUT14', 'discovered', 'Do not carry disposed businesses into later-period outcomes.'), ('S6A_T23', 'Unilever Indonesia', 'PT Unilever Indonesia Tbk continuing portfolio', 'primary_time_varying', 'continuing_operation_bridge', 'annual_report_or_financial_statement', 'FY2024–FY2025', 'Indonesia', 'STR09', 'OUT07;OUT08;OUT11;OUT12;OUT13;OUT14', 'discovered', 'Use restated/represented prior-period values when the source reports them.'), ('S6A_T24', 'Cross-company context', 'Indonesia macro and input-cost context', 'context_only', 'official_context', 'government_or_official_statistics', '2022–2025', 'Indonesia', '', '', 'conditional_not_yet_acquired', 'Acquire only when required to assess disclosed alternative factors.'), ('S6A_T25', 'Cross-company context', 'Consumer reach context', 'context_only', 'consumer_reach', 'licensed_or_public_industry_research', '2022–2025 study editions', 'Indonesia', '', 'OUT06', 'conditional_existing_limited_source', 'CRP or reach metrics remain source-native and are not market share.'), ('S6A_T26', 'Cross-company context', 'Defined category market measurement', 'context_or_conditional_comparison', 'market_measurement', 'licensed_or_public_industry_research', '2022–2025', 'Indonesia', '', 'OUT18', 'conditional_not_yet_acquired', 'Use market-share terminology only when the source explicitly measures a defined market share.')], columns=acquisition_columns)
if stage6a_acquisition_universe["target_id"].duplicated().any():
    raise RuntimeError("Duplicate acquisition target IDs found.")
print(f"Acquisition targets: {len(stage6a_acquisition_universe)}")
print(stage6a_acquisition_universe.groupby("canonical_group").size())

Acquisition targets: 26
canonical_group
Cross-company context    3
Indofood                 8
Mayora                   5
Unilever Indonesia       5
Wings Group              5
dtype: int64


## Discovered Source Inventory

Register official and inherited limited-use sources discovered during Stage 6A. Document-level extraction remains deferred to controlled acquisition.

In [5]:
source_columns = ["source_id","publisher","canonical_group","reporting_entity","document_title","source_type","source_url","direct_file_url","publication_date","reference_period","geography","reporting_scope","priority_tier","discovery_status","access_status","copyright_status","redistribution_status","automated_access_status","raw_storage_policy","claim_label","planned_role","source_notes"]
stage6a_source_inventory = pd.DataFrame([('S6ASRC_WNG_001', 'Wings Group', 'Wings Group', 'Wings Group', 'About Us', 'official_corporate_webpage', 'https://wingscorp.com/about-us/', '', '', 'current_page', 'Indonesia', 'group history, operating context and named JVs', 2, 'discovered', 'public', 'copyrighted', 'reference_only_pending_terms', 'manual_limited_only', 'reference_only', 'company_reported_current_context', 'current operating and ownership context', 'General content-reuse terms were not located during Stage 6A discovery.'), ('S6ASRC_WNG_002', 'Wings Group', 'Wings Group', 'Wings Group', 'Local Brands', 'official_corporate_webpage', 'https://wingscorp.com/local-brands/', '', '', 'current_page', 'Indonesia', 'current local consumer-brand portfolio', 2, 'discovered', 'public', 'copyrighted', 'reference_only_pending_terms', 'manual_limited_only', 'reference_only', 'company_reported_current_context', 'current portfolio context', 'Do not backdate the current portfolio page.'), ('S6ASRC_WNG_003', 'Wings Group', 'Wings Group', 'Wings Group', 'News & Activities', 'official_corporate_news_index', 'https://wingscorp.com/berita-kegiatan/', '', '', '2022–2025 archive', 'Indonesia', 'dated corporate and brand announcements', 2, 'discovered', 'public', 'copyrighted', 'reference_only_pending_terms', 'manual_limited_only', 'reference_only', 'company_reported_action', 'strategy-event discovery', 'Use article-level pages for final extraction.'), ('S6ASRC_WNG_004', 'Wings Group', 'Wings Group', 'Wings Care', 'Wings Care launches ProGuard', 'official_corporate_press_release', 'https://wingscorp.com/wings-care-luncurkan-proguard-the-next-level-of-antibacterial-body-wash-guna-lawan-mutasi-kuman-virus-dan-bakteri/', '', '2022-08-18', '2022_action', 'Indonesia', 'product launch and positioning', 2, 'discovered', 'public', 'copyrighted', 'reference_only_pending_terms', 'manual_limited_only', 'reference_only', 'company_reported_action', 'STR01/STR03 strategy event', 'Action evidence only; no effectiveness inference.'), ('S6ASRC_WNG_005', 'Wings Group', 'Wings Group', 'WINGS Food', 'Ale-Ale FunFlava Cocopandan launch', 'official_corporate_press_release', 'https://wingscorp.com/ramadhan-tiba-wings-food-luncurkan-ale-ale-funflava-cocopandan/', '', '2023-03-23', '2023_action', 'Indonesia', 'product launch and target consumer context', 2, 'discovered', 'public', 'copyrighted', 'reference_only_pending_terms', 'manual_limited_only', 'reference_only', 'company_reported_action', 'STR01/STR03 strategy event', 'Action evidence only.'), ('S6ASRC_WNG_006', 'Wings Group', 'Wings Group', 'WINGS Food', 'ISOPLUS COCO launch', 'official_corporate_press_release', 'https://wingscorp.com/wings-food-luncurkan-isoplus-coco-excellent-hydration-dengan-kesegaran-air-kelapa-muda-thailand/', '', '2023-04-02', '2023_action', 'Indonesia', 'product launch and brand positioning', 2, 'discovered', 'public', 'copyrighted', 'reference_only_pending_terms', 'manual_limited_only', 'reference_only', 'company_reported_action', 'STR01/STR03 strategy event', 'Action evidence only.'), ('S6ASRC_WNG_007', 'Wings Group', 'Wings Group', 'website', 'Privacy Policy (Glico Wings page)', 'privacy_policy', 'https://wingscorp.com/kebijakan-privasi-glico-wings/', '', '2025-03-24', 'current_terms_context', 'Indonesia', 'privacy terms; not a general copyright/reuse license', 4, 'discovered', 'public', 'copyrighted', 'not_applicable_to_content_reuse', 'manual_only', 'reference_only', 'context_only', 'legal/access context', 'Does not resolve general content redistribution rights.'), ('S6ASRC_IDF_001', 'PT Indofood Sukses Makmur Tbk', 'Indofood', 'PT Indofood Sukses Makmur Tbk', 'Annual Report', 'annual_report_index', 'https://www.indofood.com/investor-relation/annual-report', '', '', 'FY2022–FY2025', 'consolidated / segment disclosures', 'Annual reports 2022, 2023, 2024 and 2025 listed', 1, 'discovered', 'public', 'copyrighted', 'reference_only_no_repo_copy', 'manual_or_single_document_download', 'reference_only', 'company_reported', 'strategy and company-result source family', 'Resolve document-level file URLs during acquisition.'), ('S6ASRC_IDF_002', 'PT Indofood Sukses Makmur Tbk', 'Indofood', 'PT Indofood Sukses Makmur Tbk', 'Financial Statements', 'financial_statement_index', 'https://www.indofood.com/menu/financial-statements', '', '', 'FY2022–FY2025', 'consolidated', 'annual financial statements listed for target years', 1, 'discovered', 'public', 'copyrighted', 'reference_only_no_repo_copy', 'manual_or_single_document_download', 'reference_only', 'company_reported_or_audited', 'audited result source family', 'Use filed/audited definitions and notes.'), ('S6ASRC_IDF_003', 'PT Indofood Sukses Makmur Tbk', 'Indofood', 'PT Indofood Sukses Makmur Tbk', 'Indofood full-year results for 2024', 'official_earnings_release', 'https://www.indofood.com/menu/financial-press-releases/indofoods-full-year-financial-results-for-the-year-ended-31-december-2024', '', '2025-03-25', 'FY2024', 'consolidated', 'headline sales, operating profit, margin and management explanation', 2, 'discovered', 'public', 'copyrighted', 'reference_only_no_repo_copy', 'manual_limited_only', 'reference_only', 'company_reported', 'period-result corroboration', 'Reconcile with annual/audited reporting.'), ('S6ASRC_IDF_004', 'PT Indofood Sukses Makmur Tbk', 'Indofood', 'website', 'Terms & Condition', 'terms_of_use', 'https://www.indofood.com/page/terms-condition', '', '', 'current_terms', 'Global site', 'copyright and permitted-use rules', 1, 'discovered', 'public', 'copyrighted', 'limited_noncommercial_extracts_no_incorporation', 'manual_limited_only', 'reference_only', 'authoritative_legal_terms', 'legal/access audit', 'Terms permit non-commercial informational extracts but prohibit modification/incorporation/posting into another site.'), ('S6ASRC_ICBP_001', 'PT Indofood CBP Sukses Makmur Tbk', 'Indofood', 'PT Indofood CBP Sukses Makmur Tbk', 'Annual Report', 'annual_report_index', 'https://www.indofoodcbp.com/investor-relation/annual-report', '', '', 'FY2022–FY2025', 'consolidated ICBP', 'annual reports 2022–2025 listed', 1, 'discovered', 'public', 'copyrighted_presumed', 'reference_only_pending_specific_terms', 'manual_or_single_document_download', 'reference_only', 'company_reported', 'ICBP strategy and result source family', 'Specific content-reuse terms were not resolved in Stage 6A.'), ('S6ASRC_ICBP_002', 'PT Indofood CBP Sukses Makmur Tbk', 'Indofood', 'PT Indofood CBP Sukses Makmur Tbk', 'Financial Statements', 'financial_statement_index', 'https://www.indofoodcbp.com/menu/financial-statements', '', '', 'FY2022–FY2025', 'consolidated ICBP', 'financial statements for target years listed', 1, 'discovered', 'public', 'copyrighted_presumed', 'reference_only_pending_specific_terms', 'manual_or_single_document_download', 'reference_only', 'company_reported_or_audited', 'audited result source family', 'Preserve ICBP consolidation and geography.'), ('S6ASRC_ICBP_003', 'PT Indofood CBP Sukses Makmur Tbk', 'Indofood', 'PT Indofood CBP Sukses Makmur Tbk', 'ICBP full-year results for 2024', 'official_earnings_release', 'https://www.indofoodcbp.com/menu/financial-press-releases/icbps-full-year-financial-results-for-the-year-ended-31-december-2024', '', '2025-03-25', 'FY2024', 'consolidated ICBP', 'sales, operating income, margin, volume/productivity management explanation', 2, 'discovered', 'public', 'copyrighted_presumed', 'reference_only_pending_specific_terms', 'manual_limited_only', 'reference_only', 'company_reported', 'period-result and explanation source', 'Management attribution remains company-reported.'), ('S6ASRC_ICBP_004', 'PT Indofood CBP Sukses Makmur Tbk', 'Indofood', 'PT Indofood CBP Sukses Makmur Tbk', 'Consolidated Financial Statements 2025', 'audited_financial_statement', 'https://indofoodcbp.com/uploads/statement/ICBP_billingual_31_dec_25_released.pdf', '', '2026', 'FY2025', 'consolidated ICBP', 'audited consolidated financial statements', 1, 'discovered', 'public', 'copyrighted_presumed', 'reference_only_pending_specific_terms', 'single_document_download_only', 'reference_only', 'company_reported_or_audited', 'FY2025 audited results', 'Do not mirror raw PDF without explicit redistribution permission.'), ('S6ASRC_ICBP_005', 'PT Indofood CBP Sukses Makmur Tbk', 'Indofood', 'website', 'Privacy Policy', 'privacy_policy', 'https://www.indofoodcbp.com/privacy-policy', '', '', 'current_terms_context', 'website', 'privacy/submission policy only', 4, 'discovered', 'public', 'copyrighted_presumed', 'not_sufficient_for_content_reuse', 'manual_only', 'reference_only', 'context_only', 'legal/access context', 'Does not provide a general redistribution license.'), ('S6ASRC_MYR_001', 'PT Mayora Indah Tbk', 'Mayora', 'PT Mayora Indah Tbk', 'Mayora Annual Report', 'annual_report_index', 'https://www.mayoraindah.co.id/content/laporan-tahunan-mayora-21', '', '', 'FY2022–FY2025', 'consolidated', 'annual reports 2022–2025 listed', 1, 'discovered', 'public', 'copyrighted', 'no_raw_redistribution', 'manual_limited_only', 'reference_only', 'company_reported', 'strategy and company-result source family', 'Mayora terms prohibit reproduction/distribution without permission.'), ('S6ASRC_MYR_002', 'PT Mayora Indah Tbk', 'Mayora', 'PT Mayora Indah Tbk', 'Annual Financial Statements', 'financial_statement_index', 'https://mayoraindah.co.id/content/Laporan-Keuangan-Tahunan-23', '', '', 'FY2022–FY2025', 'consolidated', 'annual financial statements 2022–2025 listed', 1, 'discovered', 'public', 'copyrighted', 'no_raw_redistribution', 'manual_limited_only', 'reference_only', 'company_reported_or_audited', 'audited result source family', 'Use source-native domestic/export and accounting scope.'), ('S6ASRC_MYR_003', 'PT Mayora Indah Tbk', 'Mayora', 'PT Mayora Indah Tbk', 'Public Expose', 'public_expose_index', 'https://mayoraindah.co.id/content/Public-Expose-95', '', '', '2022 and current items', 'Indonesia', 'public-expose materials; target-period coverage has gaps', 2, 'discovered_with_period_gaps', 'public', 'copyrighted', 'no_raw_redistribution', 'manual_limited_only', 'reference_only', 'company_reported', 'strategy/disclosure discovery', 'Do not infer missing years as no strategy.'), ('S6ASRC_MYR_004', 'PT Mayora Indah Tbk', 'Mayora', 'PT Mayora Indah Tbk', 'Information Disclosure', 'regulatory_disclosure_index', 'https://mayoraindah.co.id/en/content/Keterbukaan-Informasi-91', '', '', 'current_archive', 'Indonesia / transaction-specific', 'material corporate disclosures', 1, 'discovered', 'public', 'copyrighted', 'no_raw_redistribution', 'manual_limited_only', 'reference_only', 'authoritative_filing_or_company_reported', 'ownership/capital action discovery', 'Use filing date and transaction scope.'), ('S6ASRC_MYR_005', 'PT Mayora Indah Tbk', 'Mayora', 'website', 'Terms & Conditions', 'terms_of_use', 'https://www.mayoraindah.co.id/terms-and-condition?lang=en', '', '', 'current_terms', 'website', 'copyright and site-use restrictions', 1, 'discovered', 'public', 'copyrighted', 'no_reproduction_or_systematic_download', 'no_bulk_or_systematic_download', 'reference_only', 'authoritative_legal_terms', 'legal/access audit', 'Terms prohibit reproduction, distribution, publication and systematic downloading without permission.'), ('S6ASRC_MYR_006', 'PT Mayora Indah Tbk', 'Mayora', 'PT Mayora Indah Tbk', 'Annual Report 2025', 'annual_report', 'https://mayoraindah.co.id/assets/upload/file/ar-mayora-2025-30042026-03.pdf', '', '2026', 'FY2025', 'consolidated', 'corporate structure, strategy and consolidated results', 1, 'discovered', 'public', 'copyrighted', 'no_raw_redistribution', 'single_document_reference', 'reference_only', 'company_reported', 'FY2025 annual-report reference', 'Already present in inherited source registry; do not mirror raw.'), ('S6ASRC_MYR_007', 'PT Mayora Indah Tbk', 'Mayora', 'PT Mayora Indah Tbk', 'Consolidated Financial Statements 2025', 'audited_financial_statement', 'https://www.mayoraindah.co.id/assets/upload/file/pt-mayora-indah-tbk-and-its-subsidaries-2025.pdf', '', '2026', 'FY2025', 'consolidated', 'audited financial statements and related-party classification', 1, 'discovered', 'public', 'copyrighted', 'no_raw_redistribution', 'single_document_reference', 'reference_only', 'company_reported_or_audited', 'FY2025 audited-results reference', 'Already present in inherited source registry; do not mirror raw.'), ('S6ASRC_UNV_001', 'PT Unilever Indonesia Tbk', 'Unilever Indonesia', 'PT Unilever Indonesia Tbk', 'Annual Reports', 'annual_report_index', 'https://www.unilever.co.id/en/investors/annual-financial-and-sustainability-report/annual-reports/', '', '', 'FY2022–FY2025', 'Indonesia', 'annual reports 2022–2025 with publication dates and PDF links', 1, 'discovered', 'public', 'copyrighted', 'limited_noncommercial_reproduction_no_combination', 'single_document_download', 'reference_only', 'company_reported', 'strategy and company-result source family', 'Conservative repository policy remains reference-only.'), ('S6ASRC_UNV_002', 'PT Unilever Indonesia Tbk', 'Unilever Indonesia', 'PT Unilever Indonesia Tbk', 'Annual Report 2025', 'annual_report', 'https://www.unilever.co.id/files/annual-reports-2025.pdf', '', '2026-04-29', 'FY2025', 'Indonesia', 'annual strategy, results and portfolio reporting', 1, 'discovered', 'public', 'copyrighted', 'limited_noncommercial_reproduction_no_combination', 'single_document_download', 'reference_only', 'company_reported', 'FY2025 annual-report reference', 'Do not combine or republish raw content in repository.'), ('S6ASRC_UNV_003', 'PT Unilever Indonesia Tbk', 'Unilever Indonesia', 'PT Unilever Indonesia Tbk', 'Annual Report 2024', 'annual_report', 'https://www.unilever.co.id/files/indonesia-annual-reports-2024.pdf', '', '2025-04-30', 'FY2024', 'Indonesia', 'annual strategy, results and portfolio reporting', 1, 'discovered', 'public', 'copyrighted', 'limited_noncommercial_reproduction_no_combination', 'single_document_download', 'reference_only', 'company_reported', 'FY2024 annual-report reference', 'Contains explicit strategy discussion; treat attribution as company-reported.'), ('S6ASRC_UNV_004', 'PT Unilever Indonesia Tbk', 'Unilever Indonesia', 'PT Unilever Indonesia Tbk', 'Annual Report 2023', 'annual_report', 'https://www.unilever.co.id/files/indonesia-annual-report-2023.pdf', '', '2024-05-01', 'FY2023', 'Indonesia', 'annual strategy, results and portfolio reporting', 1, 'discovered', 'public', 'copyrighted', 'limited_noncommercial_reproduction_no_combination', 'single_document_download', 'reference_only', 'company_reported', 'FY2023 annual-report reference', 'Preserve source-native definitions.'), ('S6ASRC_UNV_005', 'PT Unilever Indonesia Tbk', 'Unilever Indonesia', 'PT Unilever Indonesia Tbk', 'Annual Report 2022', 'annual_report', 'https://www.unilever.co.id/files/d5395e35-a1aa-4965-a059-828b456c10fe/unilever-ar-2022-130723-aqtsgf--1-.pdf', '', '2023-05-01', 'FY2022', 'Indonesia', 'annual strategy, results and portfolio reporting', 1, 'discovered', 'public', 'copyrighted', 'limited_noncommercial_reproduction_no_combination', 'single_document_download', 'reference_only', 'company_reported', 'FY2022 annual-report reference', 'Preserve observation-period ownership.'), ('S6ASRC_UNV_006', 'PT Unilever Indonesia Tbk', 'Unilever Indonesia', 'PT Unilever Indonesia Tbk', 'Annual Financial Statements 2025', 'audited_financial_statement', 'https://www.unilever.co.id/files/indonesia-financial-statements-q4-2025.pdf', '', '2026', 'FY2025', 'Indonesia with domestic/export split', 'audited annual financial statements; restated 2024 comparator', 1, 'discovered', 'public', 'copyrighted', 'limited_noncommercial_reproduction_no_combination', 'single_document_download', 'reference_only', 'company_reported_or_audited', 'FY2025 audited-results reference', 'Use restated/represented comparator and continuing/disposed scope.'), ('S6ASRC_UNV_007', 'PT Unilever Indonesia Tbk', 'Unilever Indonesia', 'PT Unilever Indonesia Tbk', 'Annual Financial Statements 2023', 'audited_financial_statement', 'https://www.unilever.co.id/files/unilever-indonesia-annual-financial-statements-q4-2023.pdf', '', '2024', 'FY2023 and FY2022', 'Indonesia', 'audited annual financial statements', 1, 'discovered', 'public', 'copyrighted', 'limited_noncommercial_reproduction_no_combination', 'single_document_download', 'reference_only', 'company_reported_or_audited', 'FY2023/FY2022 audited-results reference', 'Use as accounting support for both years.'), ('S6ASRC_UNV_008', 'PT Unilever Indonesia Tbk', 'Unilever Indonesia', 'PT Unilever Indonesia Tbk', 'Q1 2025 Earnings Call Presentation', 'official_investor_presentation', 'https://www.unilever.co.id/files/unvr-earnings-call-q1-2025-presentation.pdf', '', '2025', 'Q1_2025', 'Indonesia', 'strategy update including product, promotion, place and pricing actions', 1, 'discovered', 'public', 'copyrighted', 'limited_noncommercial_reproduction_no_combination', 'single_document_download', 'reference_only', 'company_reported', 'strategy evidence', 'Interim presentation is strategy evidence; Q1 outcomes remain interim context unless like-for-like.'), ('S6ASRC_UNV_009', 'PT Unilever Indonesia Tbk', 'Unilever Indonesia', 'website', 'Legal Notes', 'terms_of_use', 'https://www.unilever.co.id/en/legal/', '', '', 'current_terms', 'website', 'copyright and permitted-use rules', 1, 'discovered', 'public', 'copyrighted', 'limited_noncommercial_reproduction_no_combination', 'manual_limited_only', 'reference_only', 'authoritative_legal_terms', 'legal/access audit', 'Terms allow non-commercial informational reproduction but prohibit modification/combination with other works/sites.'), ('S6ASRC_UNV_010', 'PT Unilever Indonesia Tbk', 'Unilever Indonesia', 'PT Unilever Indonesia Tbk', 'Corporate Governance', 'official_corporate_webpage', 'https://www.unilever.co.id/en/investors/corporate-governance/', '', '', 'current_page', 'Indonesia', 'operating-company and governance context', 2, 'discovered', 'public', 'copyrighted', 'limited_noncommercial_reproduction_no_combination', 'manual_limited_only', 'reference_only', 'company_reported_current_context', 'entity-perimeter context', 'Current page only; do not backdate.'), ('S6ASRC_WPN_001', 'Worldpanel by Numerator', 'Cross-company context', 'research provider', 'Indonesia Brand Footprint 2024', 'consumer_panel_report_webpage', 'https://market.worldpanelbynumerator.com/id/News/Indonesia-Brand-Footprint-2024', '', '2024-06-28', '2024_study_edition', 'Indonesia', 'public summary of reach/consumer-choice study', 3, 'inherited_existing_limited_source', 'public_summary', 'copyrighted', 'limited_factual_use_only', 'no_bulk_extraction', 'reference_only', 'external_measurement', 'conditional consumer-reach context', 'CRP remains source-native and is not market share.'), ('S6ASRC_WPN_002', 'Worldpanel by Numerator', 'Cross-company context', 'research provider', 'Website Terms and Conditions', 'terms_of_use', 'https://market.worldpanelbynumerator.com/en/terms-and-conditions/', '', '', 'current_terms', 'website', 'intellectual-property and reuse conditions', 3, 'inherited_existing_source', 'public', 'copyrighted', 'reference_only', 'no_bulk_copy', 'reference_only', 'authoritative_legal_terms', 'legal/access audit', 'Retain provenance and limited factual derivation only.')], columns=source_columns)
stage6a_source_inventory["retrieval_date"] = RETRIEVAL_DATE
if stage6a_source_inventory["source_id"].duplicated().any():
    raise RuntimeError("Duplicate source IDs found.")
if (stage6a_source_inventory["source_url"].str.strip() == "").any():
    raise RuntimeError("Every source inventory row must preserve a source URL.")
print(f"Discovered/inherited source records: {len(stage6a_source_inventory)}")
print(stage6a_source_inventory.groupby(["canonical_group","discovery_status"]).size())

Discovered/inherited source records: 35
canonical_group        discovery_status                 
Cross-company context  inherited_existing_limited_source     1
                       inherited_existing_source             1
Indofood               discovered                            9
Mayora                 discovered                            6
                       discovered_with_period_gaps           1
Unilever Indonesia     discovered                           10
Wings Group            discovered                            7
dtype: int64


## Access, Copyright, Redistribution, and Raw-Storage Audit

Apply conservative treatment at publisher-domain level. Public accessibility does not imply permission to mirror copyrighted raw documents in the repository.

In [6]:
access_columns = ["audit_id","publisher_domain","publisher","access_status","copyright_status","terms_url","terms_review_status","redistribution_status","automated_access_status","raw_storage_policy","derived_factual_use","legal_notes"]
stage6a_access_redistribution_audit = pd.DataFrame([('S6ALEG01', 'wingscorp.com', 'Wings Group', 'public', 'copyrighted', 'https://wingscorp.com/kebijakan-privasi-glico-wings/', 'general_content_terms_not_found', 'reference_only_pending_terms', 'manual_limited_only_no_bulk_scraping', 'reference_only', 'yes_with_citation_and_narrow_factual_extraction', 'Public access is available, but the located privacy policy is not a general content-reuse license. Conservative treatment is required.'), ('S6ALEG02', 'indofood.com', 'PT Indofood Sukses Makmur Tbk', 'public', 'copyrighted', 'https://www.indofood.com/page/terms-condition', 'reviewed', 'limited_noncommercial_extracts_no_incorporation', 'manual_or_single_document_only', 'reference_only', 'yes_with_citation_and_limited_extracts', 'Terms permit non-commercial informational extracts but prohibit modification or incorporation/posting into another work or site.'), ('S6ALEG03', 'indofoodcbp.com', 'PT Indofood CBP Sukses Makmur Tbk', 'public', 'copyrighted_presumed', 'https://www.indofoodcbp.com/privacy-policy', 'specific_content_reuse_terms_not_resolved', 'reference_only_pending_specific_terms', 'manual_or_single_document_only', 'reference_only', 'yes_with_citation_and_narrow_factual_extraction', 'A privacy policy is available, but it does not provide a general redistribution license for site content or reports.'), ('S6ALEG04', 'mayoraindah.co.id', 'PT Mayora Indah Tbk', 'public', 'copyrighted', 'https://www.mayoraindah.co.id/terms-and-condition?lang=en', 'reviewed', 'no_raw_redistribution_without_permission', 'no_bulk_or_systematic_download', 'reference_only', 'yes_as_factual_reference_without_copying_protected_expression', 'Terms prohibit reproduction, distribution, publication and systematic downloading of protected site elements without permission.'), ('S6ALEG05', 'unilever.co.id', 'PT Unilever Indonesia Tbk', 'public', 'copyrighted', 'https://www.unilever.co.id/en/legal/', 'reviewed', 'limited_noncommercial_reproduction_no_combination', 'manual_or_single_document_only', 'reference_only', 'yes_with_citation_and_limited_factual_use', 'Terms allow non-commercial informational reproduction but prohibit modification or combination with other works/publications/sites.'), ('S6ALEG06', 'market.worldpanelbynumerator.com', 'Worldpanel by Numerator', 'public_summary', 'copyrighted', 'https://market.worldpanelbynumerator.com/en/terms-and-conditions/', 'inherited_reviewed', 'limited_factual_use_only', 'no_bulk_copy_or_extraction', 'reference_only', 'yes_for_limited_public_facts_only', 'Existing project source audit already limits repository use to provenance and narrow factual derivation.')], columns=access_columns)
if stage6a_access_redistribution_audit["audit_id"].duplicated().any():
    raise RuntimeError("Duplicate legal-audit IDs found.")
if (stage6a_access_redistribution_audit["raw_storage_policy"] != "reference_only").any():
    raise RuntimeError("Stage 6A must not approve copyrighted raw-source repository storage.")
print(f"Publisher-domain legal/access audits: {len(stage6a_access_redistribution_audit)}")
print(stage6a_access_redistribution_audit[["publisher_domain","terms_review_status","raw_storage_policy"]])

Publisher-domain legal/access audits: 6
                   publisher_domain  \
0                     wingscorp.com   
1                      indofood.com   
2                   indofoodcbp.com   
3                 mayoraindah.co.id   
4                    unilever.co.id   
5  market.worldpanelbynumerator.com   

                         terms_review_status raw_storage_policy  
0            general_content_terms_not_found     reference_only  
1                                   reviewed     reference_only  
2  specific_content_reuse_terms_not_resolved     reference_only  
3                                   reviewed     reference_only  
4                                   reviewed     reference_only  
5                         inherited_reviewed     reference_only  


## Acquisition Validation

Validate coverage, legal treatment, missingness semantics, entity boundaries, and stage-scope restrictions before official-source acquisition begins.

In [7]:
validation_columns = ["check_id","validation_area","check_description","result","status","critical_failure","required_treatment"]
stage6a_acquisition_validation = pd.DataFrame([('S6A001', 'input_integrity', 'All ten Stage 5 manifest-tracked inputs match their locked SHA-256 values.', '10/10 inputs passed', 'passed', 'no', 'Stop Stage 6A if any locked input differs.'), ('S6A002', 'prior_stage_gate', 'Stage 5 final design gate remains PASS_WITH_CAVEAT.', 'PASS_WITH_CAVEAT', 'passed_with_caveat', 'no', 'Carry Stage 5 caveats into acquisition.'), ('S6A003', 'focal_group_coverage', 'The acquisition universe covers all four focal groups.', '4/4 focal groups', 'passed', 'no', 'Do not add or remove focal groups after observing results.'), ('S6A004', 'primary_period', 'Primary strategy and corporate-result windows remain 2022–2025 and FY2022–FY2025.', '2022–2025 / FY2022–FY2025', 'passed', 'no', 'Keep 2026 interim evidence context-only unless like-for-like.'), ('S6A005', 'strategy_taxonomy', 'Acquisition targets use only the frozen STR01–STR09 strategy taxonomy.', '9 frozen strategy codes', 'passed', 'no', 'Do not invent post-result strategy classes.'), ('S6A006', 'outcome_taxonomy', 'Acquisition targets preserve the frozen Stage 5 outcome taxonomy.', '18 frozen outcome definitions', 'passed', 'no', 'Do not collapse unlike outcome semantics.'), ('S6A007', 'source_hierarchy', 'Official and audited sources precede secondary contextual evidence.', 'priority tiers retained', 'passed', 'no', 'Use lower-tier sources mainly for corroboration/context.'), ('S6A008', 'wings_official_strategy_sources', 'Official Wings corporate and brand announcement sources are present.', 'official web/news sources discovered', 'passed', 'no', 'Use article-level dated evidence in Stage 6B/6C.'), ('S6A009', 'wings_financial_disclosure', 'No Wings annual report or audited financial statements were found in official-site discovery.', 'not_found_in_official_site_discovery', 'passed_with_caveat', 'no', 'Treat as disclosure limitation, not weak performance or zero.'), ('S6A010', 'indofood_parent_coverage', 'Indofood parent annual-report, financial-statement and earnings-release source families are discovered.', 'covered', 'passed', 'no', 'Preserve parent/segment boundaries.'), ('S6A011', 'indofood_icbp_coverage', 'ICBP annual-report, financial-statement and earnings-release source families are discovered.', 'covered', 'passed', 'no', 'Preserve ICBP consolidated and geography scope.'), ('S6A012', 'indofood_segment_scope', 'Bogasari and consumer edible-oils/fats targets are separated from industrial/upstream operations.', 'separated', 'passed', 'no', 'Do not award portfolio-result credit to excluded operations.'), ('S6A013', 'mayora_reporting_coverage', 'Mayora annual reports and annual financial statements cover FY2022–FY2025.', 'covered', 'passed', 'no', 'Preserve consolidated and export scope.'), ('S6A014', 'mayora_sensitivity_scope', 'PT Tirta Fresindo Jaya / Le Minerale remains sensitivity-only.', 'extended_group_sensitivity_only', 'passed', 'no', 'Do not include in strict-control primary results.'), ('S6A015', 'mayora_geography', 'Domestic versus export scope is explicitly retained as an acquisition requirement.', 'retained', 'passed', 'no', 'Do not interpret mixed geography as Indonesian household demand.'), ('S6A016', 'unilever_reporting_coverage', 'Unilever annual reports cover FY2022–FY2025 and annual financial statements are identified.', 'covered', 'passed', 'no', 'Preserve restatements and accounting scope.'), ('S6A017', 'unilever_time_varying_ownership', 'Disposed businesses remain period-specific acquisition targets.', 'time-varying ownership retained', 'passed', 'no', 'Do not carry disposed businesses into later outcomes.'), ('S6A018', 'legal_access_audit', 'Content-access and redistribution treatment is recorded for each principal source domain.', '6 domains audited', 'passed_with_caveat', 'no', 'Use conservative reference-only treatment where rights are unclear or restrictive.'), ('S6A019', 'raw_storage_policy', 'No discovered copyrighted corporate report is approved for repository raw storage in Stage 6A.', 'reference_only', 'passed_with_caveat', 'no', 'Do not commit raw copyrighted PDFs absent a defensible reuse license.'), ('S6A020', 'scraping_boundary', 'No source requires prohibited or bulk scraping.', 'manual/single-document acquisition only', 'passed', 'no', 'Stop automated acquisition where terms prohibit or do not clearly permit it.'), ('S6A021', 'source_urls', 'Every discovered source-inventory record preserves a source URL.', 'complete', 'passed', 'no', 'Do not replace source URLs with local copies.'), ('S6A022', 'company_claim_labels', 'Company strategy actions and explanations retain company-reported labels.', 'retained', 'passed', 'no', 'Do not upgrade management attribution to independent causal evidence.'), ('S6A023', 'missingness', 'Not-found or undisclosed evidence is not converted to zero.', 'preserved', 'passed', 'no', 'Retain not-found, unavailable and undisclosed states explicitly.'), ('S6A024', 'no_strategy_result_inference', 'Stage 6A does not infer results from strategy or strategy from results.', 'none created', 'passed', 'no', 'Linkage belongs to later governed analysis.'), ('S6A025', 'no_cross_company_ranking', 'Stage 6A creates no cross-company performance ranking or composite score.', 'none created', 'passed', 'no', 'Keep acquisition separate from synthesis.'), ('S6A026', 'reporting_boundary', 'No analytical report or README is produced.', 'none created', 'passed', 'no', 'Defer reporting until later stages are complete.'), ('S6A027', 'stage_gate', 'The acquisition universe and source/legal audit are complete enough to begin controlled official-source acquisition.', 'PASS_WITH_CAVEAT', 'passed_with_caveat', 'no', 'Carry Wings disclosure asymmetry and conservative redistribution limits into Stage 6B.')], columns=validation_columns)
expected_checks = {f"S6A{i:03d}" for i in range(1, 28)}
if set(stage6a_acquisition_validation["check_id"]) != expected_checks:
    raise RuntimeError("Stage 6A validation registry is incomplete.")
if (stage6a_acquisition_validation["critical_failure"] == "yes").any():
    raise RuntimeError("Stage 6A contains a critical validation failure.")
required_groups = {"Wings Group","Indofood","Mayora","Unilever Indonesia"}
actual_groups = set(stage6a_acquisition_universe["canonical_group"]) & required_groups
if actual_groups != required_groups:
    raise RuntimeError("Not all four focal groups are covered in the acquisition universe.")
if stage6a_acquisition_universe.loc[stage6a_acquisition_universe["target_id"] == "S6A_T18", "portfolio_role"].squeeze() != "sensitivity_only":
    raise RuntimeError("Mayora affiliate sensitivity scope changed unexpectedly.")
if not stage6a_acquisition_universe.loc[stage6a_acquisition_universe["canonical_group"] == "Wings Group", "discovery_status"].isin(["not_found_in_official_site_discovery"]).any():
    raise RuntimeError("Wings disclosure asymmetry is not preserved.")
final_gate = stage6a_acquisition_validation.loc[stage6a_acquisition_validation["check_id"] == "S6A027"].iloc[0]
print(f"Validation checks: {len(stage6a_acquisition_validation)}")
print(f"Critical failures: {(stage6a_acquisition_validation['critical_failure'] == 'yes').sum()}")
print(f"Final gate: {final_gate['result']} / {final_gate['status']}")

Validation checks: 27
Critical failures: 0
Final gate: PASS_WITH_CAVEAT / passed_with_caveat


## Canonical Output Write

Write only the approved Stage 6A metadata artifacts. No raw/reference source directory is created at this stage.

In [8]:
OUTPUTS = {
    "metadata/stage6a_input_lock.csv": stage6a_input_lock,
    "metadata/stage6a_acquisition_universe.csv": stage6a_acquisition_universe,
    "metadata/stage6a_source_inventory.csv": stage6a_source_inventory,
    "metadata/stage6a_access_redistribution_audit.csv": stage6a_access_redistribution_audit,
    "metadata/stage6a_acquisition_validation.csv": stage6a_acquisition_validation,
}
for relative_path, dataframe in OUTPUTS.items():
    destination = OUTPUT_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(destination, index=False, lineterminator=chr(10))
print(f"Canonical Stage 6A metadata files written: {len(OUTPUTS)}")

Canonical Stage 6A metadata files written: 5


## Output Manifest

Record deterministic row counts and SHA-256 hashes for every canonical Stage 6A metadata output except the manifest itself.

In [9]:
manifest_rows = []
for relative_path, dataframe in OUTPUTS.items():
    destination = OUTPUT_ROOT / relative_path
    manifest_rows.append({"file_path": relative_path, "artifact_type": "csv", "row_count": len(dataframe), "sha256": sha256_file(destination), "locked_input_commit": INPUT_COMMIT})
stage6a_output_manifest = pd.DataFrame(manifest_rows)
manifest_path = OUTPUT_ROOT / "metadata/stage6a_output_manifest.csv"
stage6a_output_manifest.to_csv(manifest_path, index=False, lineterminator=chr(10))
if len(stage6a_output_manifest) != 5:
    raise RuntimeError("Stage 6A output manifest must track exactly five canonical metadata artifacts.")
print(f"Manifest-tracked artifacts: {len(stage6a_output_manifest)}")
print(f"Manifest path: {manifest_path}")

Manifest-tracked artifacts: 5
Manifest path: /content/fmcg_stage6a_outputs/metadata/stage6a_output_manifest.csv


## Final QA Preview

Display cardinalities, disclosure caveats, legal/raw-storage treatment, final validation gate, and output manifest for review before publication.

In [10]:
from IPython.display import Markdown, display

display(Markdown("### Stage 6A Cardinalities"))
display(pd.DataFrame([
    ("Locked Stage 5 inputs", len(stage6a_input_lock)),
    ("Acquisition targets", len(stage6a_acquisition_universe)),
    ("Source inventory records", len(stage6a_source_inventory)),
    ("Publisher-domain legal audits", len(stage6a_access_redistribution_audit)),
    ("Validation checks", len(stage6a_acquisition_validation)),
    ("Manifest-tracked artifacts", len(stage6a_output_manifest)),
], columns=["component","count"]))

display(Markdown("### Disclosure and Access Caveats"))
display(stage6a_acquisition_universe.loc[stage6a_acquisition_universe["discovery_status"].str.contains("not_found|gap|conditional", regex=True), ["target_id","canonical_group","evidence_family","discovery_status","required_treatment"]])

display(Markdown("### Raw-Storage Policy"))
display(stage6a_access_redistribution_audit[["publisher_domain","redistribution_status","automated_access_status","raw_storage_policy"]])

display(Markdown("### Final Validation Gate"))
display(stage6a_acquisition_validation.tail(5))

display(Markdown("### Output Manifest"))
display(stage6a_output_manifest)

### Stage 6A Cardinalities

,component,count
0,Locked Stage 5 inputs,10
1,Acquisition targets,26
2,Source inventory records,35
3,Publisher-domain legal audits,6
4,Validation checks,27
5,Manifest-tracked artifacts,5


### Disclosure and Access Caveats

,target_id,canonical_group,evidence_family,discovery_status,required_treatment
2,S6A_T03,Wings Group,annual_reporting,not_found_in_official_site_discovery,Retain as a disclosure limitation; do not substitute unofficial estimates as equivalent evidence.
3,S6A_T04,Wings Group,audited_financials,not_found_in_official_site_discovery,Retain as a disclosure limitation; missing financial disclosure is not zero or weak performance.
15,S6A_T16,Mayora,public_disclosures,discovered_with_period_gaps,Use only dated disclosures with explicit transaction/entity scope.
17,S6A_T18,Mayora,affiliate_context,conditional_sensitivity_only,Do not include in strict-control primary Mayora results.
23,S6A_T24,Cross-company context,official_context,conditional_not_yet_acquired,Acquire only when required to assess disclosed alternative factors.
24,S6A_T25,Cross-company context,consumer_reach,conditional_existing_limited_source,CRP or reach metrics remain source-native and are not market share.
25,S6A_T26,Cross-company context,market_measurement,conditional_not_yet_acquired,Use market-share terminology only when the source explicitly measures a defined market share.


### Raw-Storage Policy

,publisher_domain,redistribution_status,automated_access_status,raw_storage_policy
0,wingscorp.com,reference_only_pending_terms,manual_limited_only_no_bulk_scraping,reference_only
1,indofood.com,limited_noncommercial_extracts_no_incorporation,manual_or_single_document_only,reference_only
2,indofoodcbp.com,reference_only_pending_specific_terms,manual_or_single_document_only,reference_only
3,mayoraindah.co.id,no_raw_redistribution_without_permission,no_bulk_or_systematic_download,reference_only
4,unilever.co.id,limited_noncommercial_reproduction_no_combination,manual_or_single_document_only,reference_only
5,market.worldpanelbynumerator.com,limited_factual_use_only,no_bulk_copy_or_extraction,reference_only


### Final Validation Gate

,check_id,validation_area,check_description,result,status,critical_failure,required_treatment
22,S6A023,missingness,Not-found or undisclosed evidence is not converted to zero.,preserved,passed,no,"Retain not-found, unavailable and undisclosed states explicitly."
23,S6A024,no_strategy_result_inference,Stage 6A does not infer results from strategy or strategy from results.,none created,passed,no,Linkage belongs to later governed analysis.
24,S6A025,no_cross_company_ranking,Stage 6A creates no cross-company performance ranking or composite score.,none created,passed,no,Keep acquisition separate from synthesis.
25,S6A026,reporting_boundary,No analytical report or README is produced.,none created,passed,no,Defer reporting until later stages are complete.
26,S6A027,stage_gate,The acquisition universe and source/legal audit are complete enough to begin controlled official-source acquisition.,PASS_WITH_CAVEAT,passed_with_caveat,no,Carry Wings disclosure asymmetry and conservative redistribution limits into Stage 6B.


### Output Manifest

,file_path,artifact_type,row_count,sha256,locked_input_commit
0,metadata/stage6a_input_lock.csv,csv,10,8fbe53b4f36860f8edf52883c4f02c77e23bb0c9f27e79c8d22ead8b008f8877,22a5779bcd5594eb74dee162a18010121156350e
1,metadata/stage6a_acquisition_universe.csv,csv,26,8392a86c1a67106e8301a253c60f66c6afd6eca405cdaed87960482988da61eb,22a5779bcd5594eb74dee162a18010121156350e
2,metadata/stage6a_source_inventory.csv,csv,35,6b3dc1b25fa13c0c183825e9bbdc5df4e4555540ac85144aea6a79ffabbd3928,22a5779bcd5594eb74dee162a18010121156350e
3,metadata/stage6a_access_redistribution_audit.csv,csv,6,e38ebcd9f1127d4a762007dff3a0be7ea6fb5b0568beacfc8694c04615362efa,22a5779bcd5594eb74dee162a18010121156350e
4,metadata/stage6a_acquisition_validation.csv,csv,27,6eb656bc24c8b9d8dbbb0bbaf2979e429f01e5057159c8e5cc289a18dc75ffe1,22a5779bcd5594eb74dee162a18010121156350e
